   # Project 8 – Where are airports in Canada? (OSS4008A)
   Name: Nifemi Ogunmeru
   
   **Phase I:** load and clean OurAirports data, convert to an Earth Engine FeatureCollection, show an interactive map; add open Ontario aerial imagery (~20 cm) where planes are visible, a Sentinel-2 (10 m) comparison, and one function that builds the whole map for the shared dashboard.
   
   **Data:** OurAirports (https://ourairports.com/data/), public domain; Ontario Imagery Web Map Service (Land Information Ontario), Open Government Licence – Ontario; Sentinel-2 Surface Reflectance (Copernicus, via Google Earth Engine).
   
   **How to run:** replace `YOUR_PROJECT_ID` in Cell 2, then Runtime → Run all.

In [2]:
# Cell 1 – Load the libraries
import ee            # Earth Engine Python API
import geemap        # interactive Earth Engine maps (pre-installed in Colab)
import pandas as pd  # for working with tables

print('ee version:', ee.__version__, '| geemap version:', geemap.__version__)

ee version: 1.7.39 | geemap version: 0.38.9


In [3]:
# Cell 2 – Log in to Earth Engine
ee.Authenticate()
ee.Initialize(project='YOUR_PROJECT_ID')

# Tiny test: ask Earth Engine's servers to send back a word
print('Earth Engine says:', ee.String('hello').getInfo())

Earth Engine says: hello


In [4]:
# Cell 3 – Download the OurAirports data
# Source: OurAirports open data (https://ourairports.com/data/), public domain.
# We read the copy the OurAirports project keeps on GitHub (updated nightly).
URL = 'https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv'

# Read the whole CSV into a table.
# low_memory=False makes pandas read the file in one go, so each column gets one consistent type.
raw = pd.read_csv(URL, low_memory=False)

print('Downloaded on:', pd.Timestamp.now(tz='America/Toronto').date())  # Ottawa time, not UTC
print('Rows worldwide:', len(raw))
print('Columns:', raw.columns.tolist())
raw.head(3)                                          # show the first 3 rows

Downloaded on: 2026-11-07
Rows worldwide: 86345
Columns: ['id', 'ident', 'type', 'name', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'continent', 'iso_country', 'iso_region', 'municipality', 'scheduled_service', 'icao_code', 'iata_code', 'gps_code', 'local_code', 'home_link', 'wikipedia_link', 'keywords']


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN


In [5]:
# Cell 4a – Keep Canada and the 5 active facility types

# 1) Keep rows where the country code is 'CA' (Canada).
#    .copy() makes a separate table, so later edits don't affect 'raw'.
ca = raw[raw['iso_country'] == 'CA'].copy()
print('Canadian rows (all types):', len(ca))
print(ca['type'].value_counts(), '\n')    # count of each type BEFORE filtering

# 2) Keep only the 5 active types (this drops 'closed' and 'balloonport')
keep_types = ['large_airport', 'medium_airport', 'small_airport', 'heliport', 'seaplane_base']
ca = ca[ca['type'].isin(keep_types)]
print('After keeping the 5 active types:', len(ca))

Canadian rows (all types): 3369
type
small_airport     1254
closed             889
heliport           525
seaplane_base      450
medium_airport     233
large_airport       17
balloonport          1
Name: count, dtype: int64 

After keeping the 5 active types: 2479


In [6]:
# Cell 4b – Show all 'large' airports so we can check them by eye
large = ca[ca['type'] == 'large_airport']
large[['ident', 'name', 'iata_code', 'iso_region', 'municipality']]

,ident,name,iata_code,iso_region,municipality
18004,CA-1291,(Duplicate)YEG,NaN,CA-AB,Edmonton
18005,CA-1292,YEG,NaN,CA-AB,Edmonton
21216,CYEG,Edmonton International Airport,YEG,CA-AB,Edmonton
21264,CYHM,John C. Munro Hamilton International Airport,YHM,CA-ON,Hamilton
21273,CYHZ,Halifax / Stanfield International Airport,YHZ,CA-NS,Halifax
21315,CYLW,Kelowna International Airport,YLW,CA-BC,Kelowna
21349,CYOW,Ottawa Macdonald-Cartier International Airport,YOW,CA-ON,Ottawa
21374,CYQB,Quebec Jean Lesage International Airport,YQB,CA-QC,Quebec
21377,CYQG,Windsor International Airport,YQG,CA-ON,Windsor
21439,CYUL,Montreal / Pierre Elliott Trudeau Internationa...,YUL,CA-QC,Montréal


In [7]:
# Cell 4c – Remove junk rows, keep useful columns, check coordinates

# 1) Two rules that catch bad rows:
#    - the name contains "duplicate"
#    - a 'large_airport' with no IATA code (every real large Canadian airport has one)
is_duplicate  = ca['name'].str.contains('duplicate', case=False, na=False)
large_no_iata = (ca['type'] == 'large_airport') & ca['iata_code'].isna()
junk = is_duplicate | large_no_iata          # '|' means OR

# Show exactly what we remove, so a future data refresh can't silently drop a real airport
print('Removing', int(junk.sum()), 'junk rows:')
print(ca[junk][['ident', 'name', 'type', 'latitude_deg', 'longitude_deg']], '\n')
ca = ca[~junk]                               # '~' means NOT: keep everything that isn't junk

# 2) Keep only the columns we need, with shorter names
airports = ca[['ident', 'name', 'type', 'iata_code', 'icao_code', 'municipality',
               'iso_region', 'latitude_deg', 'longitude_deg', 'elevation_ft',
               'scheduled_service']].rename(columns={'latitude_deg': 'lat',
                                                     'longitude_deg': 'lon',
                                                     'iso_region': 'province'})
airports['province'] = airports['province'].str.replace('CA-', '', regex=False)  # 'CA-ON' -> 'ON'

# 3) Sanity check: keep only points inside a rough box around Canada
inside = airports['lat'].between(41, 84) & airports['lon'].between(-142, -52)
print('Points outside the Canada box:', int((~inside).sum()))
airports = airports[inside].reset_index(drop=True)

# 4) Final summary
print('Final count:', len(airports), '\n')
print(airports['type'].value_counts())

Removing 2 junk rows:
         ident            name           type  latitude_deg  longitude_deg
18004  CA-1291  (Duplicate)YEG  large_airport     53.309700    -113.580000
18005  CA-1292             YEG  large_airport     50.341652    -106.699219 

Points outside the Canada box: 0
Final count: 2477 

type
small_airport     1254
heliport           525
seaplane_base      450
medium_airport     233
large_airport       15
Name: count, dtype: int64


In [8]:
# Cell 5a – Turn the table into an Earth Engine FeatureCollection

# Earth Engine can't store NaN, so replace missing values with empty text ''
airports_ee = airports.copy()
airports_ee['elevation_ft'] = airports_ee['elevation_ft'].round()       # whole feet
airports_ee = airports_ee.astype(object).where(airports_ee.notna(), '')

# Build one ee.Feature per airport: geometry = point, properties = everything else
features = []
for row in airports_ee.to_dict('records'):             # each row as a dictionary
    lon, lat = row.pop('lon'), row.pop('lat')          # take the coordinates out of the properties
    point = ee.Geometry.Point([lon, lat])              # NOTE: longitude comes FIRST in Earth Engine
    features.append(ee.Feature(point, row))

airports_fc = ee.FeatureCollection(features)

# Check: the count should match the table, and one feature should look sensible
print('Features in the collection:', airports_fc.size().getInfo())
print('First feature:', airports_fc.first().getInfo()['properties'])

Features in the collection: 2477
First feature: {'elevation_ft': '', 'iata_code': '', 'icao_code': '', 'ident': 'CA-0002', 'municipality': 'Saint-Lazare', 'name': 'Pourvoirie Joncas Airstrip', 'province': 'QC', 'scheduled_service': 'no', 'type': 'small_airport'}


In [9]:
# Cell 5b – Interactive map: one layer per airport type, plus a legend

# Colour (hex, no '#') and legend label for each type
styles = {
    'large_airport':  {'color': 'd7191c', 'label': 'Large airports'},
    'medium_airport': {'color': 'fdae61', 'label': 'Medium airports'},
    'small_airport':  {'color': '2b83ba', 'label': 'Small airports'},
    'seaplane_base':  {'color': '5ab4ac', 'label': 'Seaplane bases'},
    'heliport':       {'color': '7b3294', 'label': 'Heliports'},
}

# Map centred on Canada. NOTE: geemap's center is [lat, lon], unlike ee.Geometry.Point([lon, lat])
Map = geemap.Map(center=[56, -96], zoom=4)

# Add layers in reverse order (heliports first, large airports last),
# because layers added later are drawn on top
for t, s in reversed(list(styles.items())):
    layer = airports_fc.filter(ee.Filter.eq('type', t))     # only airports of this type
    Map.addLayer(layer, {'color': s['color']}, s['label'],
                 shown=(t not in ['heliport', 'seaplane_base']))  # start with these 2 hidden

# Legend: label -> colour (the legend wants a '#' in front of the colour)
Map.add_legend(title='Airport types',
               legend_dict={s['label']: '#' + s['color'] for s in styles.values()})

Map   # display the map

Map(center=[56, -96], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

In [10]:
# Cell 6 – Save the cleaned table (goes in the data/ folder on GitHub)
airports.to_csv('canada_airports_clean.csv', index=False)   # index=False: don't save pandas' row numbers
print('Saved', len(airports), 'rows to canada_airports_clean.csv')

# Download the file to your computer
from google.colab import files
files.download('canada_airports_clean.csv')

Saved 2477 rows to canada_airports_clean.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Airplane view
Open Ontario aerial imagery (~20 cm per pixel) shows parked aircraft clearly. A Sentinel-2 image (10 m) of the same airport shows why free satellite imagery cannot resolve planes.

In [11]:
# Cell 7a – Airplane view: high-resolution aerial imagery of Ontario
# Source: Ontario Imagery Web Map Service, Land Information Ontario (LIO), Government of Ontario.
# Licence: Open Government Licence – Ontario (to confirm for the YOW image collection before the report).
# The service is pre-cut into map tiles. ArcGIS orders the tile address as z/y/x (zoom/row/column).
ONTARIO_TILES = ('https://ws.lioservices.lrc.gov.on.ca/arcgis2/rest/services/'
                 'LIO_Imagery/Ontario_Imagery_Web_Map_Service/MapServer/tile/{z}/{y}/{x}')

Map.add_tile_layer(url=ONTARIO_TILES,
                   name='Ontario aerial imagery (LIO)',
                   attribution='Contains information licensed under the Open Government Licence – Ontario')

# Zoom to Ottawa airport (YOW). set_center takes (lon, lat, zoom)
Map.set_center(-75.669, 45.322, 17)
Map

Map(center=[45.322, -75.669], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

In [12]:
# Cell 7b – Same airport in Sentinel-2 (10 m) to show why free satellite imagery can't see planes
yow = ee.Geometry.Point([-75.669, 45.322]).buffer(3000)    # 3 km circle around YOW (lon, lat)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')    # Sentinel-2 surface reflectance, 10 m
      .filterBounds(yow)                                   # only images covering YOW
      .filterDate('2025-06-01', '2025-09-01')              # summer 2025 (end date is exclusive)
      .sort('CLOUDY_PIXEL_PERCENTAGE')                     # clearest image first
      .first())                                            # ONE image: a median would erase moving planes

print('Sentinel-2 image date:', ee.Date(s2.get('system:time_start')).format('YYYY-MM-dd').getInfo())
print('Cloud cover (%):', s2.get('CLOUDY_PIXEL_PERCENTAGE').getInfo())

# True colour = red, green, blue bands (B4, B3, B2); values are reflectance x 10,000
Map.addLayer(s2.clip(yow), {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
             'Sentinel-2 10 m (comparison)')

Map.set_center(-75.669, 45.322, 16)    # one level out so the circle's edge is visible
Map

Sentinel-2 image date: 2025-08-15
Cloud cover (%): 0.001108


Map(center=[45.322, -75.669], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

## One function for the shared dashboard
`build_airport_map()` builds the whole map in one call, so any UI framework can display it.

In [13]:
# Cell 8 – One function that builds the full Project 8 map (this is what a dashboard will call)
def build_airport_map(fc, center=(56, -96), zoom=4, s2_compare=False):
    """
    Build the interactive airport map.
      fc         : ee.FeatureCollection of airports (from Cell 5a)
      center     : (lat, lon) the map opens on   <- geemap uses lat, lon order here
      zoom       : starting zoom level (4 = all of Canada, 16-18 = one airport)
      s2_compare : also add the Sentinel-2 10 m circle around YOW for comparison
    Returns a geemap.Map ready to display.
    """
    m = geemap.Map(center=list(center), zoom=zoom)

    # 1) Imagery FIRST, so it sits underneath the airport dots.
    #    Only switched on when zoomed in to airport level (zoom 12+); toggle it in the layers menu otherwise.
    m.add_tile_layer(url=ONTARIO_TILES, name='Ontario aerial imagery (LIO)',
                     attribution='Contains information licensed under the Open Government Licence – Ontario',
                     shown=(zoom >= 12))
    if s2_compare:
        m.addLayer(s2.clip(yow), {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
                   'Sentinel-2 10 m (comparison)', shown=False)   # off by default; switch on to compare

    # 2) Airport layers on top (same styles and order as Cell 5b)
    for t, s in reversed(list(styles.items())):
        m.addLayer(fc.filter(ee.Filter.eq('type', t)), {'color': s['color']}, s['label'],
                   shown=(t not in ['heliport', 'seaplane_base']))

    # 3) Legend
    m.add_legend(title='Airport types',
                 legend_dict={s['label']: '#' + s['color'] for s in styles.values()})
    return m



Map(center=[45.322, -75.669], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

In [14]:
# Test: open straight on Ottawa airport with the comparison layer available
yow_map = build_airport_map(airports_fc, center=(45.322, -75.669), zoom=17, s2_compare=True)
yow_map

Map(center=[45.322, -75.669], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…